# Clustering — Assignment Solutions
### PW Skills — Data Science with GenAI | Python ML Module




In [ ]:
# Common imports used throughout this notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import (make_blobs, make_moons, make_circles,
                               load_iris, load_wine, load_breast_cancer, load_digits)
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, silhouette_samples
from scipy.cluster.hierarchy import dendrogram, linkage

from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (enables 3D projection)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## Part A — Theoretical Questions

### Q1. What is unsupervised learning in the context of machine learning?

Unsupervised learning is a type of machine learning where the model is trained on data that has **no labeled output/target**. Instead of predicting a known label, the algorithm tries to discover hidden **structure, patterns, or groupings** in the data on its own — for example, clustering similar data points together, reducing dimensionality, or finding associations. Common techniques include K-Means, Hierarchical Clustering, DBSCAN, and PCA.

### Q2. How does K-Means clustering algorithm work?

K-Means works iteratively as follows:
1. Choose the number of clusters **k**.
2. Randomly initialize **k centroids**.
3. **Assignment step**: assign each data point to the nearest centroid (based on Euclidean distance).
4. **Update step**: recompute each centroid as the **mean** of all points assigned to it.
5. Repeat steps 3–4 until centroids no longer change significantly (convergence) or a maximum number of iterations is reached.

The result is k clusters that minimize the within-cluster sum of squared distances (inertia).

### Q3. Explain the concept of a dendrogram in hierarchical clustering.

A dendrogram is a **tree-like diagram** that records the sequence of merges (in agglomerative clustering) or splits (in divisive clustering) between clusters. 

- The **x-axis** typically represents individual data points/clusters.
- The **y-axis (height)** represents the distance/dissimilarity at which two clusters were merged.
- **Cutting** the dendrogram horizontally at a chosen height gives a specific number of clusters — the lower the cut, the more (smaller) clusters; the higher the cut, the fewer (larger) clusters.

It visually shows the entire clustering hierarchy without needing to pre-specify the number of clusters.

### Q4. What is the main difference between K-Means and Hierarchical Clustering?

| Aspect | K-Means | Hierarchical Clustering |
|---|---|---|
| Number of clusters | Must be specified in advance (**k**) | Not required upfront; decided by cutting the dendrogram |
| Approach | Partition-based (flat clusters), iterative | Builds a nested tree (agglomerative/divisive) |
| Scalability | Efficient, scales well to large datasets | Computationally expensive (O(n²) or more), less scalable |
| Cluster shape | Assumes roughly spherical, equally sized clusters | Can capture more complex/nested structures depending on linkage |
| Result | Single flat partition | Full hierarchy (dendrogram) that can be cut at any level |


### Q5. What are the advantages of DBSCAN over K-Means?

- **No need to specify the number of clusters** in advance — DBSCAN discovers it automatically.
- Can find **arbitrarily shaped clusters** (not just spherical/convex ones like K-Means assumes).
- **Robust to outliers/noise** — it explicitly labels noise points instead of forcing them into a cluster.
- Doesn't require clusters to be similarly sized; works well with clusters of varying density (to an extent) and shape.

### Q6. When would you use Silhouette Score in clustering?

Silhouette Score is used when:
- You want to **evaluate the quality** of a clustering result (how well-separated and cohesive the clusters are) **without ground-truth labels**.
- You want to **choose the optimal number of clusters (k)** by comparing scores across different k values.
- You want to **compare different clustering algorithms** on the same dataset to see which produces better-defined clusters.
- You want to identify **poorly clustered/misassigned individual points** using per-sample silhouette values.

### Q7. What are the limitations of Hierarchical Clustering?

- **Computationally expensive**: typically O(n²) or O(n³) time/space complexity, making it impractical for very large datasets.
- **Greedy and irreversible**: once two clusters are merged (or split), that decision cannot be undone in later steps.
- **Sensitive to noise and outliers**, which can distort the merge order.
- Choice of **linkage method and distance metric** significantly affects results, and there's no single "best" choice.
- Doesn't scale well and can be slow/memory-intensive for datasets with many samples.

### Q8. Why is feature scaling important in clustering algorithms like K-Means?

K-Means (and many clustering algorithms) rely on **distance metrics** (typically Euclidean distance) to assign points to clusters. If features have very different scales/ranges (e.g., income in thousands vs. age in years), the feature with the **larger numeric range will dominate** the distance calculation, effectively ignoring other features. **Feature scaling** (e.g., StandardScaler or MinMaxScaler) puts all features on a comparable scale so that each contributes fairly to the distance computation, leading to more meaningful clusters.

### Q9. How does DBSCAN identify noise points?

DBSCAN classifies every point as one of three types based on two parameters, **eps** (radius) and **min_samples**:
- **Core point**: has at least `min_samples` points (including itself) within distance `eps`.
- **Border point**: has fewer than `min_samples` neighbors within `eps`, but lies within the neighborhood of a core point.
- **Noise point**: does not qualify as a core point and is not within `eps` of any core point.

Points labeled as **noise** (given the label `-1` in scikit-learn) don't belong to any cluster — they are considered outliers.

### Q10. Define inertia in the context of K-Means.

**Inertia** (also called within-cluster sum of squares, WCSS) is the sum of the squared Euclidean distances between each data point and its assigned cluster centroid:

$$\text{Inertia} = \sum_{i=1}^{n} \min_{\mu_j \in C} \| x_i - \mu_j \|^2$$

Lower inertia means points are closer to their centroids (more compact clusters). K-Means tries to minimize inertia during training. However, inertia always decreases as k increases, so it alone isn't sufficient to pick the best k — hence the elbow method or silhouette score is used alongside it.

### Q11. What is the elbow method in K-Means clustering?

The elbow method is a heuristic for choosing the optimal number of clusters **k**:
1. Run K-Means for a range of k values (e.g., 1 to 10).
2. Record the **inertia** for each k.
3. Plot k vs inertia — the curve typically decreases sharply at first, then flattens out.
4. The **"elbow" point** — where the rate of decrease sharply slows down — is taken as a good estimate for the optimal k, balancing model complexity against compactness.

### Q12. Describe the concept of "density" in DBSCAN.

In DBSCAN, **density** at a point refers to the number of other data points that fall within a fixed radius `eps` around it. A region is considered "dense" if it contains at least `min_samples` points within that radius. DBSCAN groups together points that are in **densely populated regions** (connected via chains of core points), while points in **sparse, low-density regions** are treated as noise/outliers. This is why DBSCAN is called a **density-based** clustering algorithm.

### Q13. Can hierarchical clustering be used on categorical data?

**Yes.** While standard Euclidean distance doesn't make sense for categorical data, hierarchical clustering can still be applied by using an appropriate **distance/dissimilarity metric** for categorical or mixed data, such as:
- **Hamming distance** (for purely categorical/binary data),
- **Jaccard distance** (for binary/set-based features), or
- **Gower's distance** (for mixed numerical and categorical data).

As long as a valid pairwise distance/dissimilarity matrix can be computed, agglomerative clustering with an appropriate linkage method can be applied.

### Q14. What does a negative Silhouette Score indicate?

A negative silhouette value for a sample means that the sample is, on average, **closer to points in a neighboring (different) cluster than to points in its own assigned cluster**. This suggests the point has likely been **misclassified** or placed in the wrong cluster, and indicates poor/overlapping cluster structure in that region of the data.

### Q15. Explain the term "linkage criteria" in hierarchical clustering.

**Linkage criteria** define how the distance between two clusters (each possibly containing multiple points) is computed when deciding which clusters to merge next in agglomerative clustering. Common types:
- **Single linkage**: minimum distance between any pair of points in the two clusters (can create long, "chained" clusters).
- **Complete linkage**: maximum distance between any pair of points (tends to create compact, evenly sized clusters).
- **Average linkage**: average distance between all pairs of points across the two clusters.
- **Ward linkage**: merges the pair of clusters that leads to the **minimum increase in total within-cluster variance** (tends to produce balanced, compact clusters, similar in spirit to K-Means).

### Q16. Why might K-Means clustering perform poorly on data with varying cluster sizes or densities?

K-Means assumes clusters are **roughly spherical, similarly sized, and equally dense**, because it partitions space based purely on distance to centroids (Voronoi regions). When true clusters have very different **sizes or densities**:
- A large, sparse cluster may get **split** into multiple smaller K-Means clusters.
- A small, dense cluster located near a larger cluster may get **absorbed/merged** incorrectly.
- The centroid of an elongated or irregularly shaped cluster may not represent it well, causing misclassification of boundary points.

Density-based methods like DBSCAN or hierarchical clustering with appropriate linkage often handle such cases better.

### Q17. What are the core parameters in DBSCAN, and how do they influence clustering?

DBSCAN has two main parameters:
1. **`eps` (epsilon)**: the maximum radius within which to search for neighboring points.
   - Too small → most points become noise (many small/no clusters).
   - Too large → clusters merge together into one big cluster.
2. **`min_samples`**: the minimum number of points required within `eps` for a point to be considered a **core point**.
   - Higher values → requires denser regions to form a cluster, more points classified as noise.
   - Lower values → easier to form clusters, but more sensitive to noise being included.

Together, these two parameters determine what counts as "dense enough" to form a cluster, directly controlling the number, shape, and size of resulting clusters, as well as how much data is labeled as noise.

### Q18. How does K-Means++ improve upon standard K-Means initialization?

Standard K-Means randomly picks initial centroids, which can lead to **poor initialization** (e.g., centroids too close together), slow convergence, or getting stuck in a bad local optimum. **K-Means++** improves this by choosing initial centroids more intelligently:
1. Pick the first centroid randomly from the data points.
2. For each subsequent centroid, choose a point with probability **proportional to its squared distance** from the nearest already-chosen centroid (points farther from existing centroids are more likely to be picked).
3. Repeat until k centroids are selected.

This spreads out the initial centroids across the data, leading to **faster convergence** and more consistent, better-quality clustering results. It is the default initialization method (`init='k-means++'`) in scikit-learn's `KMeans`.

### Q19. What is agglomerative clustering?

Agglomerative clustering is a **bottom-up** hierarchical clustering approach:
1. Start with **each data point as its own individual cluster**.
2. Repeatedly **merge the two closest clusters** (based on a chosen distance metric and linkage criterion) into one.
3. Continue merging until all points belong to a **single cluster**, or a stopping condition (like a desired number of clusters) is reached.

The entire merging process can be visualized as a **dendrogram**, and cutting it at a chosen height/number of clusters gives the final cluster assignments.

### Q20. What makes Silhouette Score a better metric than just inertia for model evaluation?

- **Inertia** only measures how compact clusters are (within-cluster distance) — it **always decreases** as the number of clusters `k` increases (even down to 0 when k = n), so it can't be used alone to pick the best k, and it says nothing about how well-separated clusters are from each other.
- **Silhouette Score** considers **both cohesion (how close points are within their own cluster) and separation (how far they are from other clusters)**, producing a single score between -1 and 1 that doesn't trivially favor more clusters.
- Silhouette Score can also be used to compare clustering algorithms that don't produce centroids at all (like DBSCAN or hierarchical clustering), whereas inertia is specific to centroid-based methods like K-Means.

This makes Silhouette Score a more balanced and broadly applicable metric for judging clustering quality.

## Part B — Practical Questions

### Q21. Generate synthetic data with 4 centers using make_blobs and apply K-Means clustering. Visualize using a scatter plot.

In [ ]:
X_blobs, y_blobs = make_blobs(n_samples=500, centers=4, cluster_std=1.0, random_state=RANDOM_STATE)

kmeans_21 = KMeans(n_clusters=4, init='k-means++', random_state=RANDOM_STATE, n_init=10)
labels_21 = kmeans_21.fit_predict(X_blobs)

plt.figure(figsize=(6,5))
plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels_21, cmap='viridis', s=30)
plt.scatter(kmeans_21.cluster_centers_[:, 0], kmeans_21.cluster_centers_[:, 1],
            c='red', marker='X', s=200, label='Centroids')
plt.title("K-Means Clustering on Synthetic Data (4 centers)")
plt.legend()
plt.show()


### Q22. Load the Iris dataset and use Agglomerative Clustering to group the data into 3 clusters. Display the first 10 predicted labels.

In [ ]:
iris = load_iris()
X_iris = iris.data

agg_22 = AgglomerativeClustering(n_clusters=3)
labels_22 = agg_22.fit_predict(X_iris)

print("First 10 predicted labels:", labels_22[:10])


### Q23. Generate synthetic data using make_moons and apply DBSCAN. Highlight outliers in the plot.

In [ ]:
X_moons, _ = make_moons(n_samples=300, noise=0.07, random_state=RANDOM_STATE)

dbscan_23 = DBSCAN(eps=0.2, min_samples=5)
labels_23 = dbscan_23.fit_predict(X_moons)

plt.figure(figsize=(6,5))
core_mask = labels_23 != -1
plt.scatter(X_moons[core_mask, 0], X_moons[core_mask, 1], c=labels_23[core_mask], cmap='viridis', s=30, label='Clustered')
plt.scatter(X_moons[~core_mask, 0], X_moons[~core_mask, 1], c='red', marker='x', s=60, label='Outliers/Noise')
plt.title("DBSCAN Clustering on make_moons (outliers highlighted)")
plt.legend()
plt.show()


### Q24. Load the Wine dataset and apply K-Means clustering after standardizing the features. Print the size of each cluster.

In [ ]:
wine = load_wine()
X_wine = wine.data

scaler_24 = StandardScaler()
X_wine_scaled = scaler_24.fit_transform(X_wine)

kmeans_24 = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10)
labels_24 = kmeans_24.fit_predict(X_wine_scaled)

cluster_sizes = pd.Series(labels_24).value_counts().sort_index()
print("Cluster sizes:\n", cluster_sizes)


### Q25. Use make_circles to generate synthetic data and cluster it using DBSCAN. Plot the result.

In [ ]:
X_circles, _ = make_circles(n_samples=400, noise=0.05, factor=0.5, random_state=RANDOM_STATE)

dbscan_25 = DBSCAN(eps=0.15, min_samples=5)
labels_25 = dbscan_25.fit_predict(X_circles)

plt.figure(figsize=(6,5))
plt.scatter(X_circles[:, 0], X_circles[:, 1], c=labels_25, cmap='viridis', s=30)
plt.title("DBSCAN Clustering on make_circles")
plt.show()


### Q26. Load the Breast Cancer dataset, apply MinMaxScaler, and use K-Means with 2 clusters. Output the cluster centroids.

In [ ]:
cancer = load_breast_cancer()
X_cancer = cancer.data

minmax_26 = MinMaxScaler()
X_cancer_scaled = minmax_26.fit_transform(X_cancer)

kmeans_26 = KMeans(n_clusters=2, random_state=RANDOM_STATE, n_init=10)
kmeans_26.fit(X_cancer_scaled)

centroids_df = pd.DataFrame(kmeans_26.cluster_centers_, columns=cancer.feature_names)
print("Cluster Centroids (scaled features):")
centroids_df


### Q27. Generate synthetic data using make_blobs with varying cluster standard deviations and cluster with DBSCAN.

In [ ]:
X_varstd, _ = make_blobs(n_samples=500, centers=3, cluster_std=[0.5, 1.5, 2.5], random_state=RANDOM_STATE)

dbscan_27 = DBSCAN(eps=0.8, min_samples=5)
labels_27 = dbscan_27.fit_predict(X_varstd)

plt.figure(figsize=(6,5))
plt.scatter(X_varstd[:, 0], X_varstd[:, 1], c=labels_27, cmap='viridis', s=30)
plt.title("DBSCAN on Blobs with Varying Standard Deviations")
plt.show()

n_clusters = len(set(labels_27)) - (1 if -1 in labels_27 else 0)
print("Number of clusters found:", n_clusters)
print("Number of noise points:", list(labels_27).count(-1))


### Q28. Load the Digits dataset, reduce it to 2D using PCA, and visualize clusters from K-Means.

In [ ]:
digits = load_digits()
X_digits = digits.data

pca_28 = PCA(n_components=2, random_state=RANDOM_STATE)
X_digits_pca = pca_28.fit_transform(X_digits)

kmeans_28 = KMeans(n_clusters=10, random_state=RANDOM_STATE, n_init=10)
labels_28 = kmeans_28.fit_predict(X_digits_pca)

plt.figure(figsize=(7,6))
scatter = plt.scatter(X_digits_pca[:, 0], X_digits_pca[:, 1], c=labels_28, cmap='tab10', s=15)
plt.colorbar(scatter, label='Cluster')
plt.title("K-Means Clusters on Digits Dataset (PCA-reduced to 2D)")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.show()


### Q29. Create synthetic data using make_blobs and evaluate silhouette scores for k = 2 to 5. Display as a bar chart.

In [ ]:
X_29, _ = make_blobs(n_samples=500, centers=4, random_state=RANDOM_STATE)

k_values = range(2, 6)
sil_scores = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_29)
    score = silhouette_score(X_29, labels)
    sil_scores.append(score)
    print(f"k={k} -> Silhouette Score: {score:.4f}")

plt.figure(figsize=(6,4))
plt.bar([str(k) for k in k_values], sil_scores, color='teal')
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Scores for k = 2 to 5")
plt.show()


### Q30. Load the Iris dataset and use hierarchical clustering to group data. Plot a dendrogram with average linkage.

In [ ]:
X_iris_30 = load_iris().data

linked_30 = linkage(X_iris_30, method='average')

plt.figure(figsize=(10,5))
dendrogram(linked_30)
plt.title("Dendrogram - Iris Dataset (Average Linkage)")
plt.xlabel("Sample Index")
plt.ylabel("Distance")
plt.show()


### Q31. Generate synthetic data with overlapping clusters using make_blobs, then apply K-Means and visualize with decision boundaries.

In [ ]:
X_31, _ = make_blobs(n_samples=500, centers=3, cluster_std=2.5, random_state=RANDOM_STATE)

kmeans_31 = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10)
kmeans_31.fit(X_31)

# Create a mesh grid to visualize decision boundaries
h = 0.05
x_min, x_max = X_31[:, 0].min() - 1, X_31[:, 0].max() + 1
y_min, y_max = X_31[:, 1].min() - 1, X_31[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

Z = kmeans_31.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(7,6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
plt.scatter(X_31[:, 0], X_31[:, 1], c=kmeans_31.labels_, cmap='viridis', edgecolor='k', s=30)
plt.scatter(kmeans_31.cluster_centers_[:, 0], kmeans_31.cluster_centers_[:, 1],
            c='red', marker='X', s=200, label='Centroids')
plt.title("K-Means Decision Boundaries on Overlapping Clusters")
plt.legend()
plt.show()


### Q32. Load the Digits dataset and apply DBSCAN after reducing dimensions with t-SNE. Visualize the results.

In [ ]:
X_digits_32 = load_digits().data

tsne_32 = TSNE(n_components=2, random_state=RANDOM_STATE, perplexity=30)
X_digits_tsne = tsne_32.fit_transform(X_digits_32)

dbscan_32 = DBSCAN(eps=3, min_samples=5)
labels_32 = dbscan_32.fit_predict(X_digits_tsne)

plt.figure(figsize=(7,6))
scatter = plt.scatter(X_digits_tsne[:, 0], X_digits_tsne[:, 1], c=labels_32, cmap='tab10', s=15)
plt.colorbar(scatter, label='Cluster')
plt.title("DBSCAN on Digits Dataset (t-SNE reduced)")
plt.show()

print("Number of clusters:", len(set(labels_32)) - (1 if -1 in labels_32 else 0))
print("Number of noise points:", list(labels_32).count(-1))


### Q33. Generate synthetic data using make_blobs and apply Agglomerative Clustering with complete linkage. Plot the result.

In [ ]:
X_33, _ = make_blobs(n_samples=400, centers=4, random_state=RANDOM_STATE)

agg_33 = AgglomerativeClustering(n_clusters=4, linkage='complete')
labels_33 = agg_33.fit_predict(X_33)

plt.figure(figsize=(6,5))
plt.scatter(X_33[:, 0], X_33[:, 1], c=labels_33, cmap='viridis', s=30)
plt.title("Agglomerative Clustering (Complete Linkage)")
plt.show()


### Q34. Load the Breast Cancer dataset and compare inertia values for K = 2 to 6 using K-Means. Show results in a line plot.

In [ ]:
X_cancer_34 = StandardScaler().fit_transform(load_breast_cancer().data)

k_range_34 = range(2, 7)
inertias_34 = []

for k in k_range_34:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km.fit(X_cancer_34)
    inertias_34.append(km.inertia_)
    print(f"k={k} -> Inertia: {km.inertia_:.2f}")

plt.figure(figsize=(6,4))
plt.plot(list(k_range_34), inertias_34, marker='o')
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Inertia vs K - Breast Cancer Dataset")
plt.grid(True)
plt.show()


### Q35. Generate synthetic concentric circles using make_circles and cluster using Agglomerative Clustering with single linkage.

In [ ]:
X_circ_35, _ = make_circles(n_samples=400, noise=0.03, factor=0.5, random_state=RANDOM_STATE)

agg_35 = AgglomerativeClustering(n_clusters=2, linkage='single')
labels_35 = agg_35.fit_predict(X_circ_35)

plt.figure(figsize=(6,5))
plt.scatter(X_circ_35[:, 0], X_circ_35[:, 1], c=labels_35, cmap='viridis', s=30)
plt.title("Agglomerative Clustering (Single Linkage) on Concentric Circles")
plt.show()


### Q36. Use the Wine dataset, apply DBSCAN after scaling the data, and count the number of clusters (excluding noise).

In [ ]:
X_wine_36 = StandardScaler().fit_transform(load_wine().data)

dbscan_36 = DBSCAN(eps=2.0, min_samples=5)
labels_36 = dbscan_36.fit_predict(X_wine_36)

n_clusters_36 = len(set(labels_36)) - (1 if -1 in labels_36 else 0)
n_noise_36 = list(labels_36).count(-1)

print("Number of clusters (excluding noise):", n_clusters_36)
print("Number of noise points:", n_noise_36)


### Q37. Generate synthetic data with make_blobs and apply KMeans. Then plot the cluster centers on top of the data points.

In [ ]:
X_37, _ = make_blobs(n_samples=400, centers=5, random_state=RANDOM_STATE)

kmeans_37 = KMeans(n_clusters=5, random_state=RANDOM_STATE, n_init=10)
labels_37 = kmeans_37.fit_predict(X_37)

plt.figure(figsize=(6,5))
plt.scatter(X_37[:, 0], X_37[:, 1], c=labels_37, cmap='viridis', s=30, alpha=0.6)
plt.scatter(kmeans_37.cluster_centers_[:, 0], kmeans_37.cluster_centers_[:, 1],
            c='red', marker='X', s=250, edgecolor='black', label='Cluster Centers')
plt.title("K-Means Clustering with Cluster Centers")
plt.legend()
plt.show()


### Q38. Load the Iris dataset, cluster with DBSCAN, and print how many samples were identified as noise.

In [ ]:
X_iris_38 = load_iris().data

dbscan_38 = DBSCAN(eps=0.5, min_samples=5)
labels_38 = dbscan_38.fit_predict(X_iris_38)

n_noise_38 = list(labels_38).count(-1)
print("Number of samples identified as noise:", n_noise_38)
print("Number of clusters found:", len(set(labels_38)) - (1 if -1 in labels_38 else 0))


### Q39. Generate synthetic non-linearly separable data using make_moons, apply K-Means, and visualize the clustering result.

In [ ]:
X_moons_39, _ = make_moons(n_samples=300, noise=0.07, random_state=RANDOM_STATE)

kmeans_39 = KMeans(n_clusters=2, random_state=RANDOM_STATE, n_init=10)
labels_39 = kmeans_39.fit_predict(X_moons_39)

plt.figure(figsize=(6,5))
plt.scatter(X_moons_39[:, 0], X_moons_39[:, 1], c=labels_39, cmap='viridis', s=30)
plt.scatter(kmeans_39.cluster_centers_[:, 0], kmeans_39.cluster_centers_[:, 1],
            c='red', marker='X', s=200, label='Centroids')
plt.title("K-Means on make_moons (non-linearly separable data)")
plt.legend()
plt.show()
print("Note: K-Means struggles here since it assumes spherical clusters, unlike the crescent moon shapes.")


### Q40. Load the Digits dataset, apply PCA to reduce to 3 components, then use KMeans and visualize with a 3D scatter plot.

In [ ]:
X_digits_40 = load_digits().data

pca_40 = PCA(n_components=3, random_state=RANDOM_STATE)
X_digits_pca3 = pca_40.fit_transform(X_digits_40)

kmeans_40 = KMeans(n_clusters=10, random_state=RANDOM_STATE, n_init=10)
labels_40 = kmeans_40.fit_predict(X_digits_pca3)

fig = plt.figure(figsize=(8,7))
ax = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(X_digits_pca3[:, 0], X_digits_pca3[:, 1], X_digits_pca3[:, 2],
                      c=labels_40, cmap='tab10', s=15)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
ax.set_title("K-Means Clusters on Digits Dataset (PCA - 3 Components)")
plt.colorbar(scatter, label='Cluster', shrink=0.6)
plt.show()


### Q41. Generate synthetic blobs with 5 centers and apply KMeans. Then use silhouette_score to evaluate the clustering.

In [ ]:
X_41, _ = make_blobs(n_samples=500, centers=5, random_state=RANDOM_STATE)

kmeans_41 = KMeans(n_clusters=5, random_state=RANDOM_STATE, n_init=10)
labels_41 = kmeans_41.fit_predict(X_41)

score_41 = silhouette_score(X_41, labels_41)
print("Silhouette Score for K-Means (5 clusters):", score_41)


### Q42. Load the Breast Cancer dataset, reduce dimensionality using PCA, and apply Agglomerative Clustering. Visualize in 2D.

In [ ]:
X_cancer_42 = StandardScaler().fit_transform(load_breast_cancer().data)

pca_42 = PCA(n_components=2, random_state=RANDOM_STATE)
X_cancer_pca = pca_42.fit_transform(X_cancer_42)

agg_42 = AgglomerativeClustering(n_clusters=2)
labels_42 = agg_42.fit_predict(X_cancer_pca)

plt.figure(figsize=(6,5))
plt.scatter(X_cancer_pca[:, 0], X_cancer_pca[:, 1], c=labels_42, cmap='viridis', s=25)
plt.title("Agglomerative Clustering on Breast Cancer Dataset (PCA - 2D)")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.show()


### Q43. Generate noisy circular data using make_circles and visualize clustering results from KMeans and DBSCAN side-by-side.

In [ ]:
X_circ_43, _ = make_circles(n_samples=400, noise=0.06, factor=0.5, random_state=RANDOM_STATE)

kmeans_43 = KMeans(n_clusters=2, random_state=RANDOM_STATE, n_init=10)
labels_km_43 = kmeans_43.fit_predict(X_circ_43)

dbscan_43 = DBSCAN(eps=0.15, min_samples=5)
labels_db_43 = dbscan_43.fit_predict(X_circ_43)

fig, axes = plt.subplots(1, 2, figsize=(12,5))

axes[0].scatter(X_circ_43[:, 0], X_circ_43[:, 1], c=labels_km_43, cmap='viridis', s=25)
axes[0].set_title("K-Means Clustering")

axes[1].scatter(X_circ_43[:, 0], X_circ_43[:, 1], c=labels_db_43, cmap='viridis', s=25)
axes[1].set_title("DBSCAN Clustering")

plt.suptitle("K-Means vs DBSCAN on Noisy Circular Data")
plt.show()


### Q44. Load the Iris dataset and plot the Silhouette Coefficient for each sample after KMeans clustering.

In [ ]:
X_iris_44 = load_iris().data

kmeans_44 = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10)
labels_44 = kmeans_44.fit_predict(X_iris_44)

sample_silhouette_44 = silhouette_samples(X_iris_44, labels_44)

plt.figure(figsize=(7,5))
y_lower = 10
for i in range(3):
    ith_cluster_sil = sample_silhouette_44[labels_44 == i]
    ith_cluster_sil.sort()
    size_cluster_i = ith_cluster_sil.shape[0]
    y_upper = y_lower + size_cluster_i
    plt.fill_betweenx(np.arange(y_lower, y_upper), 0, ith_cluster_sil, alpha=0.7, label=f'Cluster {i}')
    y_lower = y_upper + 10

plt.axvline(x=silhouette_score(X_iris_44, labels_44), color='red', linestyle='--', label='Average Score')
plt.xlabel("Silhouette Coefficient")
plt.ylabel("Sample Index (per cluster)")
plt.title("Silhouette Plot - Iris Dataset (K-Means, k=3)")
plt.legend()
plt.show()


### Q45. Generate synthetic data using make_blobs and apply Agglomerative Clustering with 'average' linkage. Visualize clusters.

In [ ]:
X_45, _ = make_blobs(n_samples=400, centers=4, random_state=RANDOM_STATE)

agg_45 = AgglomerativeClustering(n_clusters=4, linkage='average')
labels_45 = agg_45.fit_predict(X_45)

plt.figure(figsize=(6,5))
plt.scatter(X_45[:, 0], X_45[:, 1], c=labels_45, cmap='viridis', s=30)
plt.title("Agglomerative Clustering (Average Linkage)")
plt.show()


### Q46. Load the Wine dataset, apply KMeans, and visualize the cluster assignments in a seaborn pairplot (first 4 features).

In [ ]:
wine_46 = load_wine()
X_wine_46 = StandardScaler().fit_transform(wine_46.data)

kmeans_46 = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10)
labels_46 = kmeans_46.fit_predict(X_wine_46)

df_wine_46 = pd.DataFrame(wine_46.data[:, :4], columns=wine_46.feature_names[:4])
df_wine_46['Cluster'] = labels_46.astype(str)

sns.pairplot(df_wine_46, hue='Cluster', palette='viridis')
plt.suptitle("K-Means Cluster Assignments - Wine Dataset (First 4 Features)", y=1.02)
plt.show()


### Q47. Generate noisy blobs using make_blobs and use DBSCAN to identify both clusters and noise points. Print the count.

In [ ]:
X_47, _ = make_blobs(n_samples=500, centers=3, cluster_std=1.5, random_state=RANDOM_STATE)
# add some random noise points
rng = np.random.RandomState(RANDOM_STATE)
noise_pts = rng.uniform(low=X_47.min()-5, high=X_47.max()+5, size=(30, 2))
X_47_noisy = np.vstack([X_47, noise_pts])

dbscan_47 = DBSCAN(eps=0.8, min_samples=5)
labels_47 = dbscan_47.fit_predict(X_47_noisy)

n_clusters_47 = len(set(labels_47)) - (1 if -1 in labels_47 else 0)
n_noise_47 = list(labels_47).count(-1)

print("Number of clusters identified:", n_clusters_47)
print("Number of noise points identified:", n_noise_47)

plt.figure(figsize=(6,5))
plt.scatter(X_47_noisy[:, 0], X_47_noisy[:, 1], c=labels_47, cmap='viridis', s=25)
plt.title("DBSCAN - Clusters and Noise on Noisy Blobs")
plt.show()


### Q48. Load the Digits dataset, reduce dimensions using t-SNE, then apply Agglomerative Clustering and plot the clusters.

In [ ]:
X_digits_48 = load_digits().data

tsne_48 = TSNE(n_components=2, random_state=RANDOM_STATE, perplexity=30)
X_digits_tsne_48 = tsne_48.fit_transform(X_digits_48)

agg_48 = AgglomerativeClustering(n_clusters=10)
labels_48 = agg_48.fit_predict(X_digits_tsne_48)

plt.figure(figsize=(7,6))
scatter = plt.scatter(X_digits_tsne_48[:, 0], X_digits_tsne_48[:, 1], c=labels_48, cmap='tab10', s=15)
plt.colorbar(scatter, label='Cluster')
plt.title("Agglomerative Clustering on Digits Dataset (t-SNE reduced)")
plt.show()


## Summary

This notebook covered:
- **20 theoretical questions** explaining unsupervised learning, K-Means, Hierarchical/Agglomerative Clustering, DBSCAN, linkage criteria, inertia, the elbow method, and Silhouette Score.
- **28 practical questions** implementing K-Means, Agglomerative Clustering (single/complete/average linkage), and DBSCAN on synthetic datasets (`make_blobs`, `make_moons`, `make_circles`) and real datasets (Iris, Wine, Breast Cancer, Digits), along with dimensionality reduction (PCA, t-SNE), scaling (StandardScaler, MinMaxScaler), dendrograms, silhouette analysis, and 2D/3D visualizations.
